In [1]:
import pandas as pd
import sys
sys.path.insert(0, '..')
from config import settings

report = pd.read_csv(settings.OUTPUT_DIR / 'validation_report.csv')
clean = pd.read_csv(settings.OUTPUT_DIR / 'valid_transactions.csv')
flagged = pd.read_csv(settings.OUTPUT_DIR / 'flagged_issues.csv')

## 2 Currency look up table

In [2]:
from config.constants import CURRENCY_INFO

# Build a DataFrame from the dictionary
currency_df = pd.DataFrame([
    {'code': code, **info}
    for code, info in CURRENCY_INFO.items()
])
currency_df

,code,decimals,name,symbol
0,USD,2,US Dollar,$
1,EUR,2,Euro,€
2,GBP,2,British Pound,£
3,JPY,0,Japanese Yen,¥
4,CHF,2,Swiss Franc,CHF
5,CAD,2,Canadian Dollar,C$
6,AUD,2,Australian Dollar,A$
7,CNY,2,Chinese Yuan,¥
8,KRW,0,Korean Won,₩
9,TWD,2,Taiwan Dollar,NT$


## 3 Merge

In [3]:
# Merge: add currency details to every transaction

merged = report.merge(currency_df, left_on='currency', right_on='code')
print(f"Before: {report.shape}")
print(f"After:  {merged.shape}")
merged[['transaction_id','currency', 'name', 'symbol', 'decimals']].head()

Before: (200, 49)
After:  (200, 53)


,transaction_id,currency,name,symbol,decimals
0,LC-ZCOTOH,GBP,British Pound,£,2
1,LC-OH9SDB,JPY,Japanese Yen,¥,0
2,LC-6GVWRD,USD,US Dollar,$,2
3,LC-0OXFQ8,USD,US Dollar,$,2
4,LC-Q2Y9V8,GBP,British Pound,£,2


## 4  But what if a row doesn't match? -> error code lookup table from validation_rules

In [4]:
# First, let's build an error code lookup table from validation_rules
from config.validation_rules import ALL_ERROR_CODES

error_lookup = pd.DataFrame([
    {'error_code': code, 'message': info['message'], 'severity': info['severity']}
    for code, info in ALL_ERROR_CODES.items()

])
print(f"Error lookup: {len(error_lookup)} codes")
error_lookup.head()


Error lookup: 54 codes


,error_code,message,severity
0,AMT001,Amount zero,CRITICAL
1,AMT002,Amount negative,CRITICAL
2,AMT003,Amount exceeds maximum allowed,HIGH
3,AMT004,Amount below minimum allowed,HIGH
4,AMT005,Invalid amount format,CRITICAL


## 5

In [5]:
# Step 1: Explode error codes into individual rows
errors_series= flagged[['transaction_id', 'error_codes']].copy()
errors_exploded = errors_series.assign(
    error_code = errors_series['error_codes'].str.split(', ')
).explode('error_code')

errors_exploded[['transaction_id', 'error_code']].head(10)




,transaction_id,error_code
0,LC-0OXFQ8,LEI004
0,LC-0OXFQ8,DATE007
1,LC-Q2Y9V8,LEI004
1,LC-Q2Y9V8,SHIP002
2,LC-5LXO6Q,LEI004
3,LC-V55PSQ,DATE002
4,LC-DTJVZH,LEI004
4,LC-DTJVZH,XVAL002
5,LC-QIMQOD,LEI004
6,LC-UZFK8U,LEI004


## 6

In [6]:
# Step 2: Merge with error lookup to get messages and severity
# Merge: add severity and message to each error

errors_enriched = errors_exploded.merge(error_lookup, on='error_code')
print(f"Exploded: {len(errors_exploded)} rows")
print(f"Enriched: {len(errors_enriched)} rows")
errors_enriched[['transaction_id', 'error_code', 'severity', 'message']].head(10)

Exploded: 237 rows
Enriched: 237 rows


,transaction_id,error_code,severity,message
0,LC-0OXFQ8,LEI004,HIGH,LEI not found in GLEIF.md database
1,LC-0OXFQ8,DATE007,MEDIUM,LC validity period exceeds maximum
2,LC-Q2Y9V8,LEI004,HIGH,LEI not found in GLEIF.md database
3,LC-Q2Y9V8,SHIP002,HIGH,Port of loading missing
4,LC-5LXO6Q,LEI004,HIGH,LEI not found in GLEIF.md database
5,LC-V55PSQ,DATE002,HIGH,Date invalid format
6,LC-DTJVZH,LEI004,HIGH,LEI not found in GLEIF.md database
7,LC-DTJVZH,XVAL002,MEDIUM,Issuing bank country does not match applicant ...
8,LC-QIMQOD,LEI004,HIGH,LEI not found in GLEIF.md database
9,LC-UZFK8U,LEI004,HIGH,LEI not found in GLEIF.md database


## 7

In [7]:
# Create a SMALL lookup — intentionally missing some codes
small_lookup = error_lookup.query("severity == 'CRITICAL'")
print(f"Full lookup: {len(error_lookup)} codes")
print(f"Small lookup: {len(small_lookup)} codes (CRITICAL only)")

# Now merge with the small lookup
test_merge = errors_exploded.merge(small_lookup, on='error_code')
print(f"\nExploded errors: {len(errors_exploded)} rows")
print(f"After merge:     {len(test_merge)} rows")

Full lookup: 54 codes
Small lookup: 15 codes (CRITICAL only)

Exploded errors: 237 rows
After merge:     45 rows


## 8

In [8]:
# LEFT merge — keep ALL errors, add severity where possible
test_left = errors_exploded.merge(small_lookup, on='error_code', how='left')
print(f"After LEFT merge: {len(test_left)} rows")
test_left[['transaction_id', 'error_code', 'severity', 'message']].head(10)

# How many got matched vs unmatched?
print(test_left['severity'].value_counts(dropna=False))


After LEFT merge: 237 rows
severity
NaN         192
CRITICAL     45
Name: count, dtype: int64


In [9]:
# This tells pandas: "left should have many rows per key, right should have ONE"
# One more critical parameter. What if your lookup table has duplicate keys? That can silently multiply your rows.
# Pandas has a safety net:

errors_enriched = errors_exploded.merge(
    error_lookup,
    on='error_code',
    how='left',
    validate='many_to_one'
)

print(f"Rows: {len(errors_enriched)}")

Rows: 237


## 10 DAY 55

## 9

In [10]:
# This tells pandas: "left should have many rows per key, right should have ONE"
errors_enriched = errors_exploded.merge(
    error_lookup,
    on='error_code',
    how='left',
    validate='many_to_one'
)
print(f"Rows: {len(errors_enriched)}")

Rows: 237


## ================= DAY 55 ==========================

## 10  Multi-Key Merge

In [11]:
# Sometimes one column isn't enough to make a unique match. Imagine you have error statistics broken down by both currency AND severity:

# Build a summary: avg error count per currency + validation status
currency_status_stats = (
    report
    .groupby(['currency', 'validation_status'])['error_count']
    .agg(['mean', 'count'])
    .round(2)
    .reset_index()
)
currency_status_stats.columns = ['currency', 'validation_status', 'avg_errors', 'num_transactions']
currency_status_stats





,currency,validation_status,avg_errors,num_transactions
0,EUR,CLEAN,0.00,17
1,EUR,FLAGGED,1.48,27
2,GBP,CLEAN,0.00,19
3,GBP,FLAGGED,2.39,28
4,JPY,CLEAN,0.00,21
5,JPY,FLAGGED,1.57,37
6,USD,CLEAN,0.00,15
7,USD,FLAGGED,2.00,36


## 11

In [12]:
# Merge on TWO columns — currency AND validation_status must BOTH match
enriched = report.merge(
    currency_status_stats,
    on = ['currency', 'validation_status'],
    how='left',
    validate='many_to_one'
)
print(f"Rows: {len(enriched)}")
enriched[['transaction_id', 'currency', 'validation_status', 'avg_errors', 'num_transactions']].head(10)

# on= takes a list when you need multiple keys. Pandas will only connect rows where both keys match. This is super common when you have different "groups" in your data (e.g. currency + validation status) and your lookup table has stats for each group. Always check your merge results to make sure you got the expected number of rows!

# # When column names are DIFFERENT:
# report.merge(stats, left_on=['currency', 'validation_status'],
#                     right_on=['cur_code', 'val_status'])


Rows: 200


,transaction_id,currency,validation_status,avg_errors,num_transactions
0,LC-ZCOTOH,GBP,CLEAN,0.00,19
1,LC-OH9SDB,JPY,CLEAN,0.00,21
2,LC-6GVWRD,USD,CLEAN,0.00,15
3,LC-0OXFQ8,USD,FLAGGED,2.00,36
4,LC-Q2Y9V8,GBP,FLAGGED,2.39,28
5,LC-L2W37D,EUR,CLEAN,0.00,17
6,LC-5LXO6Q,EUR,FLAGGED,1.48,27
7,LC-TOGN1W,USD,CLEAN,0.00,15
8,LC-3J2D54,GBP,CLEAN,0.00,19
9,LC-V55PSQ,JPY,FLAGGED,1.57,37


# 12  Suffixes — When Column Names Collide

In [13]:
# Now a different problem. What happens when both tables have a column with the same name that is NOT the merge key?

# Build two summary tables that BOTH have a column called 'avg_errors'
# key isimleri AYNI ama islevleri FARKLI
by_currency = (
    report.groupby('currency')['error_count']
    .mean().round(2)
    .reset_index()
    .rename(columns={'error_count': 'avg_errors'})
)

by_status = (
    report.groupby('validation_status')['error_count']
    .mean().round(2)
    .reset_index()
    .rename(columns={'error_count': 'avg_errors'})
)

print("By currency:")
print(by_currency)
print("\nBy status:")
print(by_status)




By currency:
  currency  avg_errors
0      EUR        0.91
1      GBP        1.43
2      JPY        1.00
3      USD        1.41

By status:
  validation_status  avg_errors
0             CLEAN        0.00
1           FLAGGED        1.85


## 13  MERGE


In [14]:
# First merge — no conflict yet
step1 = report.merge(by_currency, on='currency', how='left')

# Second merge — BOTH tables now have 'avg_errors'!
step2 = step1.merge(by_status, on='validation_status', how='left')

step2[['transaction_id', 'currency', 'validation_status', 'avg_errors_x', 'avg_errors_y']].head(5)


,transaction_id,currency,validation_status,avg_errors_x,avg_errors_y
0,LC-ZCOTOH,GBP,CLEAN,1.43,0.00
1,LC-OH9SDB,JPY,CLEAN,1.00,0.00
2,LC-6GVWRD,USD,CLEAN,1.41,0.00
3,LC-0OXFQ8,USD,FLAGGED,1.41,1.85
4,LC-Q2Y9V8,GBP,FLAGGED,1.43,1.85


## 14 Suffixes


In [15]:
# Pandas automatically adds suffixes to distinguish the two 'avg_errors' columns. By default it uses _x and _y, but you can customize this:
step1 = report.merge(by_currency, on='currency', how='left', suffixes=('', '_by_currency'))
step2 = step1.merge(by_status, on='validation_status', how='left', suffixes=('', '_by_status'))

#step2[['transaction_id', 'currency', 'validation_status', 'avg_errors_by_currency', 'avg_errors_by_status']].head(5)


# 15

In [16]:
# Step 1: Build both summary tables with CLEAR names from the start
by_currency = (
    report.groupby('currency')['error_count']
    .mean().round(2)
    .reset_index()
    .rename(columns={'error_count': 'avg_errors_by_currency'})
)

by_status = (
    report.groupby('validation_status')['error_count']
    .mean().round(2)
    .reset_index()
    .rename(columns={'error_count': 'avg_errors_by_status'})
)

# Step 2: Merge both — no conflicts, no suffixes needed
enriched = (
    report
    .merge(by_currency, on='currency', how='left', validate='many_to_one')
    .merge(by_status, on='validation_status', how='left', validate='many_to_one')
)

enriched[['transaction_id', 'currency', 'validation_status', 'avg_errors_by_currency', 'avg_errors_by_status']].head(5)


,transaction_id,currency,validation_status,avg_errors_by_currency,avg_errors_by_status
0,LC-ZCOTOH,GBP,CLEAN,1.43,0.00
1,LC-OH9SDB,JPY,CLEAN,1.00,0.00
2,LC-6GVWRD,USD,CLEAN,1.41,0.00
3,LC-0OXFQ8,USD,FLAGGED,1.41,1.85
4,LC-Q2Y9V8,GBP,FLAGGED,1.43,1.85


CONCAT()

In [17]:
# Stack clean and flagged back together
recombined = pd.concat([clean, flagged])
print(f"Clean:      {len(clean)} rows")
print(f"Flagged:    {len(flagged)} rows")
print(f"Recombined: {len(recombined)} rows")
print(f"Report:     {len(report)} rows")

recombined.index.tolist()

recombined = pd.concat([clean, flagged], ignore_index=True)
recombined.index.tolist()[-5:]  # check the end

Clean:      72 rows
Flagged:    128 rows
Recombined: 200 rows
Report:     200 rows


[195, 196, 197, 198, 199]

In [18]:
# Step 1: Build both summary tables with CLEAR names from the start
by_currency = (
    report.groupby('currency')['error_count']
    .mean().round(2)
    .reset_index()
    .rename(columns={'error_count': 'avg_errors_by_currency'})
)

by_status = (
    report.groupby('validation_status')['error_count']
    .mean().round(2)
    .reset_index()
    .rename(columns={'error_count': 'avg_errors_by_status'})
)

# Step 2: Merge both — no conflicts, no suffixes needed
enriched = (
    report
    .merge(by_currency, on='currency', how='left', validate='many_to_one')
    .merge(by_status, on='validation_status', how='left', validate='many_to_one')
)

enriched[['transaction_id', 'currency', 'validation_status', 'avg_errors_by_currency', 'avg_errors_by_status']].head(5)

,transaction_id,currency,validation_status,avg_errors_by_currency,avg_errors_by_status
0,LC-ZCOTOH,GBP,CLEAN,1.43,0.00
1,LC-OH9SDB,JPY,CLEAN,1.00,0.00
2,LC-6GVWRD,USD,CLEAN,1.41,0.00
3,LC-0OXFQ8,USD,FLAGGED,1.41,1.85
4,LC-Q2Y9V8,GBP,FLAGGED,1.43,1.85


In [19]:
# Add a source column BEFORE concatenating
clean_tagged = clean.assign(source='clean')
flagged_tagged = flagged.assign(source='flagged')

recombined_tagged = pd.concat([clean_tagged, flagged_tagged], ignore_index=True)
recombined_tagged['source'].value_counts()

source
flagged    128
clean       72
Name: count, dtype: int64

##  Day 56 — Pivot Tables.


In [20]:
# Your first pivot table
pd.pivot_table(
    report,
    values='error_count',
    index='currency',
    columns='validation_status',
    aggfunc='count'
)

validation_status,CLEAN,FLAGGED
currency,,
EUR,17,27
GBP,19,28
JPY,21,37
USD,15,36


In [21]:
# Pivot with totals and mean error count
pd.pivot_table(
    report,
    values='error_count',
    index='currency',
    columns='validation_status',
    aggfunc='mean',
    margins=True,
    margins_name='Overall'

).round(2)

validation_status,CLEAN,FLAGGED,Overall
currency,,,
EUR,0.0,1.48,0.91
GBP,0.0,2.39,1.43
JPY,0.0,1.57,1.00
USD,0.0,2.00,1.41
Overall,0.0,1.85,1.18


## MultiIndex on columns

In [22]:
pd.pivot_table(
    report,
    values='error_count',
    index='currency',
    columns='validation_status',
    aggfunc=['count', 'mean']
).round(2)



count          mean        
validation_status CLEAN FLAGGED CLEAN FLAGGED
currency                                     
EUR                  17      27   0.0    1.48
GBP                  19      28   0.0    2.39
JPY                  21      37   0.0    1.57
USD                  15      36   0.0    2.00

In [23]:
pivot = pd.pivot_table(
    report,
    values='error_count',
    index='currency',
    columns='validation_status',
    aggfunc=['count', 'mean']
).round(2)

# Flatten multi-level columns: ('count', 'CLEAN') → 'count_CLEAN'
pivot.columns = ['_'.join(col) for col in pivot.columns]
pivot

,count_CLEAN,count_FLAGGED,mean_CLEAN,mean_FLAGGED
currency,,,,
EUR,17,27,0.0,1.48
GBP,19,28,0.0,2.39
JPY,21,37,0.0,1.57
USD,15,36,0.0,2.00


Pivot Table with Multiple Values

In [24]:
pd.pivot_table(
    report,
    values=['error_count', 'tolerance_percent'],
    index='currency',
    aggfunc={'error_count': 'mean', 'tolerance_percent': 'mean'} # aggfunc is a dict — different function per column
).round(2)

,error_count,tolerance_percent
currency,,
EUR,0.91,5.00
GBP,1.43,4.47
JPY,1.00,5.09
USD,1.41,4.51


In [25]:
# GroupBy version — same result, long format
report.groupby('currency').agg(
    error_count=('error_count', 'mean'),
    tolerance_percent=('tolerance_percent', 'mean')
).round(2)

,error_count,tolerance_percent
currency,,
EUR,0.91,5.00
GBP,1.43,4.47
JPY,1.00,5.09
USD,1.41,4.51


##  Day 57 — Melt & Stack/Unstack.

Part 1: pd.melt() — The Basics  And notice the full journey: melt() → groupby() → pivot_table(). You went wide → long → analyzed → wide again. Full circle.

In [26]:
# Melt: turn applicant_country and beneficiary_country into one column
countries_long = pd.melt(
    report,
    id_vars=['transaction_id'],              # KEEP this as identifier
    value_vars=['applicant_country', 'beneficiary_country'],  # STACK these
    var_name='party_role',
    value_name='country'
)
print(f"Report rows: {len(report)}")
print(f"Melted rows: {len(countries_long)}")
countries_long.head(6)

Report rows: 200
Melted rows: 400


,transaction_id,party_role,country
0,LC-ZCOTOH,applicant_country,JP
1,LC-OH9SDB,applicant_country,US
2,LC-6GVWRD,applicant_country,US
3,LC-0OXFQ8,applicant_country,KR
4,LC-Q2Y9V8,applicant_country,TW
5,LC-L2W37D,applicant_country,JP


In [27]:
# Clean up the party_role labels
countries_long['party_role'] = countries_long['party_role'].str.replace('_country', '')

# Now: which countries appear most across ALL parties?
countries_long['country'].value_counts().head(10)

country
US    149
DE     53
KR     50
GB     47
JP     42
TW     32
CH     26
XX      1
Name: count, dtype: int64

In [28]:
# Breakdown by role AND country
countries_long.groupby(['party_role', 'country']).size().reset_index(name='count').pivot_table(
    index='country',
    columns='party_role',
    values='count',
    fill_value=0
).sort_values('applicant', ascending=False)

party_role,applicant,beneficiary
country,,
US,81.0,68.0
KR,28.0,22.0
DE,26.0,27.0
JP,22.0,20.0
CH,16.0,10.0
GB,15.0,32.0
TW,11.0,21.0
XX,1.0,0.0


In [29]:
# STEP 1- Groups by two columns at once and counts how many rows fall in each combination:
countries_long.groupby(['party_role', 'country']).size()

# .size() = count of rows per group. Returns a Series with a MultiIndex.

party_role   country
applicant    CH         16
             DE         26
             GB         15
             JP         22
             KR         28
             TW         11
             US         81
             XX          1
beneficiary  CH         10
             DE         27
             GB         32
             JP         20
             KR         22
             TW         21
             US         68
dtype: int64

In [30]:
# STEP 2 - Flattens the MultiIndex Series back into a regular DataFrame, naming the counts column 'count'

# turning the Series into a DataFrame
countries_long.groupby(['party_role', 'country']).size().reset_index( ) #  The count column is named 0 by default

# So you use name='count' to rename it
countries_long.groupby(['party_role', 'country']).size().reset_index(name='count')

# name='count' is just what to call the new column that holds the .size() numbers.

,party_role,country,count
0,applicant,CH,16
1,applicant,DE,26
2,applicant,GB,15
3,applicant,JP,22
4,applicant,KR,28
5,applicant,TW,11
6,applicant,US,81
7,applicant,XX,1
8,beneficiary,CH,10
9,beneficiary,DE,27


In [31]:
# STEP 3 - Reshapes from long format → wide format. Each unique party_role value becomes its own column:
countries_long.groupby(['party_role', 'country']).size().reset_index(name='count').pivot_table(index='country', columns='party_role', values='count', fill_value=0)

#  fill_value=0 — if a combination didn't exist (e.g. a country had no applicants), put 0 instead of NaN.

party_role,applicant,beneficiary
country,,
CH,16.0,10.0
DE,26.0,27.0
GB,15.0,32.0
JP,22.0,20.0
KR,28.0,22.0
TW,11.0,21.0
US,81.0,68.0
XX,1.0,0.0


In [32]:
# STEP 4 -  Sorts rows by the applicant column, highest first:
countries_long.groupby(['party_role', 'country']).size().reset_index(name='count').pivot_table(index='country', columns='party_role', values='count', fill_value=0).sort_values('applicant', ascending=False)


party_role,applicant,beneficiary
country,,
US,81.0,68.0
KR,28.0,22.0
DE,26.0,27.0
JP,22.0,20.0
CH,16.0,10.0
GB,15.0,32.0
TW,11.0,21.0
XX,1.0,0.0


## Part 2: Stack & Unstack

melt works on regular columns. stack/unstack work on the index. They're the index-based version of the same idea.

In [33]:
# Start with a pivot table (wide format)
pivot = pd.pivot_table(
    report,
    values='error_count',
    index='currency',
    columns='validation_status',
    aggfunc='count'

)

print("WIDE (pivot):")
print(pivot)

# Stack: wide → long (columns become index level)
stacked = pivot.stack()
print("\nLONG (stacked):")
print(stacked)

WIDE (pivot):
validation_status  CLEAN  FLAGGED
currency                         
EUR                   17       27
GBP                   19       28
JPY                   21       37
USD                   15       36

LONG (stacked):
currency  validation_status
EUR       CLEAN                17
          FLAGGED              27
GBP       CLEAN                19
          FLAGGED              28
JPY       CLEAN                21
          FLAGGED              37
USD       CLEAN                15
          FLAGGED              36
dtype: int64


Same data, different shape. .stack() took the column headers (CLEAN, FLAGGED) and pushed them down into the index

— now you have a MultiIndex with two levels: currency + validation_status.

And the reverse — .unstack() pulls an index level back up to columns:

In [34]:
# Unstack: long → wide (index level becomes columns again)
stacked.unstack()

validation_status,CLEAN,FLAGGED
currency,,
EUR,17,27
GBP,19,28
JPY,21,37
USD,15,36


# ================== DAY 58 Apply & Map ================

In [35]:
# Way 1: Dictionary mapping (you know this!)
severity_map = {'CRITICAL': 4, 'HIGH': 3, 'MEDIUM': 2, 'LOW': 1, 'NONE': 0}
report['severity_score'] = report['severity'].map(severity_map)
report[['transaction_id', 'severity', 'severity_score']].head(5)

,transaction_id,severity,severity_score
0,LC-ZCOTOH,NONE,0
1,LC-OH9SDB,NONE,0
2,LC-6GVWRD,NONE,0
3,LC-0OXFQ8,HIGH,3
4,LC-Q2Y9V8,HIGH,3


In [36]:
# Way 2: Function mapping
report['currency_lower'] = report['currency'].map(str.lower)

# Way 3: Lambda mapping
report['amount_label'] = report['error_count'].map(lambda x: 'clean' if x== 0 else 'has_errors')
# x = 0 → "x is assigned 0" or "x gets 0"
# x == 0 → "x equals 0" or "x is equal to 0"

# lambda x: 'clean' if x== 0 else 'has_errors
# lambda x: → "for each x" or "given x"
# "Given x — if x equals 0, return 'clean', otherwise return 'has_errors'."


report[['transaction_id', 'currency', 'currency_lower', 'error_count', 'amount_label']].head(5)

,transaction_id,currency,currency_lower,error_count,amount_label
0,LC-ZCOTOH,GBP,gbp,0,clean
1,LC-OH9SDB,JPY,jpy,0,clean
2,LC-6GVWRD,USD,usd,0,clean
3,LC-0OXFQ8,USD,usd,2,has_errors
4,LC-Q2Y9V8,GBP,gbp,2,has_errors


Part 2: .apply() on a Series — When lambda gets complex

.map() is great for simple one-liners. But what if your logic needs multiple steps? That's where .apply() with a real function shines:

In [37]:
# Classify transactions by risk level based on error count
def classify_risk(error_count):
    """Multi-level classification — too complex for a lambda."""
    if error_count == 0:
        return 'no_risk'
    elif error_count <=2:
        return 'low_risk'
    elif error_count <=4:
        return 'medium_risk'
    else:
       return 'high_risk'

  # apply calls classify_risk on each value:
# classify_risk(0)  → 'no_risk'
# result:
# report['risk_level']  →  ['no_risk', 'low_risk', 'high_risk', 'medium_risk', 'no_risk', 'low_risk', ...]
report['risk_level'] = report['error_count'].apply(classify_risk)
report['risk_level'].value_counts()

# .apply(func) calls the function once for each value in the Series, and collects the results into a new Series.
# .apply(func) = "run this function on every value in the column, give me back the results as a new column."


risk_level
low_risk       103
no_risk         72
medium_risk     18
high_risk        7
Name: count, dtype: int64

Part 3: .apply() on a DataFrame — Row-level logic

This is the big one. Sometimes you need data from multiple columns to compute one result. That's apply(axis=1) — it sends an entire row to your function:

In [38]:
# Create a risk summary that uses MULTIPLE columns
def transaction_summary(row):
    """Build a human-readable summary from multiple fields."""
    status = row['validation_status']
    currency = row['currency']
    errors = row['error_count']

    if status == 'CLEAN':
        return f"{currency} transaction — clean"
    else:
        return f"{currency} transaction — {errors} error(s), {row['severity']} severity"

# ("apply this function to every row")
report['summary'] = report.apply(transaction_summary, axis=1) # give me one full row at a time
#  The results are collected into a new Series, which gets assigned back as the 'summary' column.
report[['transaction_id', 'summary']].head() # (double brackets = list of column names)
# The key difference: axis=1 means "apply this function to each row." Without it, pandas sends each column.

,transaction_id,summary
0,LC-ZCOTOH,GBP transaction — clean
1,LC-OH9SDB,JPY transaction — clean
2,LC-6GVWRD,USD transaction — clean
3,LC-0OXFQ8,"USD transaction — 2 error(s), HIGH severity"
4,LC-Q2Y9V8,"GBP transaction — 2 error(s), HIGH severity"


| axis= | Sends to function | Think of it as |
|-------|------------------|----------------|
| axis=0 (default) | Each column | Working vertically |
| axis=1 | Each row | Working horizontally |

 axis=0 collapses rows. axis=1 collapses columns.

In [39]:
# The apply way (what you just did) — works but slow
# report.apply(transaction_summary, axis=1)

# The vectorized way — same result, MUCH faster

import numpy as np


report['summary_v2'] = np.where(
    report['validation_status'] == 'CLEAN',        # condition  (True/False for every row)
    report['currency'] + ' transaction - clean',   # use this if True
    report['currency'] + ' transaction - ' + report['error_count'].astype(str) + ' error(s) ' + report['severity'] + ' severity'   # use this if False

)
# NumPy sends the entire column (200 values) to a C function that processes them all in one shot — no Python function call per row, no loop visible or hidden
report[['summary', 'summary_v2']].head()

#   Use np.where() when your logic is a simple true/false split on column values:
#   np.where(df['amount'] > 1000, 'high', 'low')
#   np.where() evaluates all three arguments before checking the condition —
#   the true-branch and false-branch are both computed for every row, then it picks which  to use.



,summary,summary_v2
0,GBP transaction — clean,GBP transaction - clean
1,JPY transaction — clean,JPY transaction - clean
2,USD transaction — clean,USD transaction - clean
3,"USD transaction — 2 error(s), HIGH severity",USD transaction - 2 error(s) HIGH severity
4,"GBP transaction — 2 error(s), HIGH severity",GBP transaction - 2 error(s) HIGH severity


| Rows | .apply(axis=1) | np.where() |
|------|----------------|------------|
| 200 | ~5ms | ~1ms |
| 200,000 | ~5 seconds | ~50ms |
| 2,000,000 | ~50 seconds | ~500ms |

| Situation | Use | Why |
|-----------|-----|-----|
| Simple if/else on columns | np.where() | Vectorized, fast |
| Multiple conditions | np.select() | Vectorized multi-branch |
| Complex logic needing many columns | .apply(axis=1) | Flexibility over speed |
| One column, simple transform | .map() | Simplest tool |

np.select() — the vectorized version of your classify_risk

In [40]:
# Your apply version:
# report['error_count'].apply(classify_risk)

# Vectorized version with np.select:

conditions = [
    report['error_count'] == 0,
    report['error_count'] <= 2,
    report['error_count'] <= 4,

]
choices = ['no_risk', 'low_risk', 'medium_risk']

report['risk_v2'] = np.select(conditions, choices, default= 'high_risk')
report[['error_count', 'risk_level', 'risk_v2']].head()

# np.select() takes a list of conditions and a list of results — first matching condition wins. The default catches everything else
# first matching condition wins.
# It's like np.where() but for more than two outcomes — a vectorized if/elif/elif/else.

#  why default='high_risk' exists — it's the catch-all for anything that didn't match any condition.
#  Without it, numpy would fill unmatched rows with 0 (its default default).















,error_count,risk_level,risk_v2
0,0,no_risk,no_risk
1,0,no_risk,no_risk
2,0,no_risk,no_risk
3,2,low_risk,low_risk
4,2,low_risk,low_risk




  ---
  How it evaluates — row by row mentally

  For each row, it checks conditions in order, and uses the first one that is True:

  error_count == 0  → True?  use 'no_risk'    ✓ stop here
  error_count <= 2  → True?  use 'low_risk'   ✓ stop here
  error_count <= 4  → True?  use 'medium_risk'✓ stop here
  none matched?     →        use default 'high_risk'

  Equivalent plain Python

  if error_count == 0:
      return 'no_risk'
  elif error_count <= 2:
      return 'low_risk'
  elif error_count <= 4:
      return 'medium_risk'
  else:
      return 'high_risk'   # ← this is the default

  default is literally the else branch.


# ========== Day 59 — MultiIndex.============

currency  validation_status
EUR       CLEAN                17
          FLAGGED              27
GBP       CLEAN                19
          FLAGGED              28

That's a MultiIndex — two levels of row labels instead of one. It's like a filing cabinet: the first drawer is currency, inside each drawer are folders for validation_status.

GroupBy and pivot tables create MultiIndexes all the time. If you don't understand them, you'll keep calling reset_index() to escape.

Part 1: Creating a MultiIndex


In [41]:
# GroupBy with two levels — DON'T reset_index this time
multi = report.groupby(['currency', 'validation_status'])['error_count'].agg(['count', 'mean']).round(2)
print(type(multi.index))
print(multi.index)
print(multi)



<class 'pandas.MultiIndex'>
MultiIndex([('EUR',   'CLEAN'),
            ('EUR', 'FLAGGED'),
            ('GBP',   'CLEAN'),
            ('GBP', 'FLAGGED'),
            ('JPY',   'CLEAN'),
            ('JPY', 'FLAGGED'),
            ('USD',   'CLEAN'),
            ('USD', 'FLAGGED')],
           names=['currency', 'validation_status'])
                            count  mean
currency validation_status             
EUR      CLEAN                 17  0.00
         FLAGGED               27  1.48
GBP      CLEAN                 19  0.00
         FLAGGED               28  2.39
JPY      CLEAN                 21  0.00
         FLAGGED               37  1.57
USD      CLEAN                 15  0.00
         FLAGGED               36  2.00


Part 2: Accessing Data in a MultiIndex
This is where it gets tricky. Regular .loc[] works differently now:

In [42]:
# Level 1: Get ALL data for one currency
print("ALL EUR")          ## all EUR rows
print(multi.loc['EUR'])
print()

# Both levels: Get ONE specific cell
print("EUR + FLAGGED:")
print(multi.loc[('EUR', 'FLAGGED')])  ## exactly EUR + CLEAN row
# print(multi.loc[('EUR', 'FLAGGED'), 'mean'])  ## EUR + CLEAN row, only the 'mean' column  -> 1,48

## The key: with MultiIndex, .loc[] accepts tuples to specify multiple levels

ALL EUR
                   count  mean
validation_status             
CLEAN                 17  0.00
FLAGGED               27  1.48

EUR + FLAGGED:
count    27.00
mean      1.48
Name: (EUR, FLAGGED), dtype: float64


  Regular Index — you already know this

  df.loc[3]          # row where index == 3
  df.loc['EUR']      # row where index == 'EUR'

  One value → finds one row.


what if you want all FLAGGED across all currencies?  You need xs():

In [ ]:
%%sql


In [43]:
# Cross-section: get all FLAGGED regardless of currency
print("All FLAGGED:")
print(multi.xs('FLAGGED', level='validation_status')) # "Give me all rows where validation_status == 'FLAGGED', regardless of currency."

## xs() lets you slice a MultiIndex by any level by name, without needing to know its position.

All FLAGGED:
          count  mean
currency             
EUR          27  1.48
GBP          28  2.39
JPY          37  1.57
USD          36  2.00


| What you want | Code | Returns |
|--------------|------|---------|
| All of one top-level group | .loc['EUR'] | DataFrame |
| One specific cell | .loc[('EUR', 'FLAGGED')] | Series |
| Slice across second level | .xs('FLAGGED', level='validation_status') | DataFrame |
| Slice across first level | .xs('EUR', level='currency') | DataFrame |

In [44]:
# Sort by mean error count, descending
multi.sort_values('mean', ascending=False)


,,count,mean
currency,validation_status,,
GBP,FLAGGED,28,2.39
USD,FLAGGED,36,2.00
JPY,FLAGGED,37,1.57
EUR,FLAGGED,27,1.48
GBP,CLEAN,19,0.00
EUR,CLEAN,17,0.00
JPY,CLEAN,21,0.00
USD,CLEAN,15,0.00


Part 4: Rearranging Levels
What if you want validation_status as the first level instead of currency?

In [45]:
# Swap the levels
swapped = multi.swaplevel()
swapped








,,count,mean
validation_status,currency,,
CLEAN,EUR,17,0.00
FLAGGED,EUR,27,1.48
CLEAN,GBP,19,0.00
FLAGGED,GBP,28,2.39
CLEAN,JPY,21,0.00
FLAGGED,JPY,37,1.57
CLEAN,USD,15,0.00
FLAGGED,USD,36,2.00


Levels swapped — validation_status is now the outer level, currency is inner. But notice CLEAN and FLAGGED are alternating instead of grouped. Fix that with:

In [46]:
swapped.sort_index()

count  mean
validation_status currency             
CLEAN             EUR          17  0.00
                  GBP          19  0.00
                  JPY          21  0.00
                  USD          15  0.00
FLAGGED           EUR          27  1.48
                  GBP          28  2.39
                  JPY          37  1.57
                  USD          36  2.00

Part 5: Escaping the MultiIndex — reset_index()
Sometimes you just want a regular DataFrame back. You've used this before but now you understand what it's resetting:

In [47]:
# Back to flat DataFrame
flat = multi.reset_index()
flat

,currency,validation_status,count,mean
0,EUR,CLEAN,17,0.00
1,EUR,FLAGGED,27,1.48
2,GBP,CLEAN,19,0.00
3,GBP,FLAGGED,28,2.39
4,JPY,CLEAN,21,0.00
5,JPY,FLAGGED,37,1.57
6,USD,CLEAN,15,0.00
7,USD,FLAGGED,36,2.00
